In [1]:
# !pip install youtube-transcript-api yt-dlp faster-whisper requests

In [2]:
import os
import sys
import shutil
from typing import List, Dict, Optional, Tuple
from datetime import datetime
import json
import time

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
import yt_dlp

In [3]:
def extract_youtube_transcript_direct(youtube_url: str) -> Dict:
    """
    Extract transcript directly from YouTube using youtube-transcript-api.
    
    Args:
        youtube_url: Full YouTube URL or video ID
        
    Returns:
        Dictionary with transcript, metadata, and status
    """
    
    def get_video_id(url: str) -> str:
        """Extract video ID from YouTube URL."""
        if len(url) == 11:
            return url
        if 'youtube.com' in url:
            return url.split('v=')[1].split('&')[0]
        if 'youtu.be' in url:
            return url.split('/')[-1].split('?')[0]
        return url
    
    try:
        video_id = get_video_id(youtube_url)
        
        # Attempt to get available transcripts (version compatibility)
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        except AttributeError:
            transcript_list = YouTubeTranscriptApi().list(video_id)
        except TypeError as exc:
            if 'required positional argument' not in str(exc):
                raise
            transcript_list = YouTubeTranscriptApi().list_transcripts(video_id)
        
        # Try to get English transcript first
        try:
            transcript = transcript_list.find_transcript(['en'])
        except NoTranscriptFound:
            # Fallback to any available transcript
            transcript = transcript_list.find_transcript(['en', 'es', 'fr', 'de'])
        
        # Get transcript entries
        transcript_data = transcript.fetch()
        
        # Normalize entries for dict/attribute compatibility
        transcript_entries = []
        for entry in transcript_data:
            if isinstance(entry, dict):
                transcript_entries.append(entry)
            else:
                transcript_entries.append({
                    'text': getattr(entry, 'text', ''),
                    'start': getattr(entry, 'start', 0.0),
                    'duration': getattr(entry, 'duration', 0.0),
                })
        
        # Parse transcript
        full_text = ' '.join([entry['text'] for entry in transcript_entries if entry.get('text')])
        duration_seconds = 0.0
        if transcript_entries:
            last_entry = transcript_entries[-1]
            duration_seconds = last_entry.get('start', 0.0) + last_entry.get('duration', 0.0)
        
        result = {
            'status': 'success',
            'method': 'youtube-transcript-api',
            'video_id': video_id,
            'transcript': full_text,
            'entries': transcript_entries,
            'entry_count': len(transcript_entries),
            'duration_seconds': duration_seconds,
            'language': transcript.language,
            'is_auto_generated': transcript.is_generated,
            'extracted_at': datetime.now().isoformat()
        }
        
        return result
        
    except VideoUnavailable:
        return {
            'status': 'error',
            'error_type': 'VideoUnavailable',
            'message': f'Video {youtube_url} is not available (private/deleted)',
            'url': youtube_url
        }
    except TranscriptsDisabled:
        return {
            'status': 'error',
            'error_type': 'TranscriptsDisabled',
            'message': 'Transcripts are disabled for this video',
            'url': youtube_url
        }
    except NoTranscriptFound:
        return {
            'status': 'fallback_required',
            'error_type': 'NoTranscriptFound',
            'message': 'No transcripts available via API',
            'url': youtube_url,
            'next_step': 'Use yt-dlp + faster-whisper fallback'
        }
    except Exception as e:
        return {
            'status': 'error',
            'error_type': type(e).__name__,
            'message': str(e),
            'url': youtube_url
        }

In [4]:
# Test with public videos
test_youtube_urls = [
    'https://www.youtube.com/watch?v=dQw4w9WgXcQ',  # Rick Roll (widely available)
    'https://www.youtube.com/watch?v=jNQXAC9IVRw',  # Me at the zoo (first YouTube video)
]
 
print("=" * 80)
print("TEST 1: YouTube Transcript Extraction (Direct Method)")
print("=" * 80)
 
for url in test_youtube_urls:
    print(f"\nProcessing: {url}")
    result = extract_youtube_transcript_direct(url)
    
    if result['status'] == 'success':
        print(f"✅ Success")
        print(f"   - Video ID: {result['video_id']}")
        print(f"   - Entries: {result['entry_count']}")
        print(f"   - Duration: {result['duration_seconds']:.1f} seconds")
        print(f"   - Language: {result['language']}")
        print(f"   - Auto-generated: {result['is_auto_generated']}")
        print(f"   - Transcript preview: {result['transcript'][:150]}...")
    else:
        print(f"❌ {result['error_type']}: {result['message']}")

TEST 1: YouTube Transcript Extraction (Direct Method)

Processing: https://www.youtube.com/watch?v=dQw4w9WgXcQ
✅ Success
   - Video ID: dQw4w9WgXcQ
   - Entries: 61
   - Duration: 211.3 seconds
   - Language: English
   - Auto-generated: False
   - Transcript preview: [♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any ...

Processing: https://www.youtube.com/watch?v=jNQXAC9IVRw
✅ Success
   - Video ID: jNQXAC9IVRw
   - Entries: 6
   - Duration: 18.9 seconds
   - Language: English
   - Auto-generated: False
   - Transcript preview: All right, so here we are, in front of the
elephants the cool thing about these guys is that they
have really... really really long trunks and that's ...


In [12]:
def download_instagram_reel_audio(instagram_url: str, output_path: str = '/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp') -> Dict:
    """
    Download audio from Instagram Reel using yt-dlp.
    
    Args:
        instagram_url: Instagram Reel URL
        output_path: Where to save audio file
        
    Returns:
        Dictionary with download status and audio file path
    """
    
    os.makedirs(output_path, exist_ok=True)
    
    # yt-dlp options
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(output_path, '%(id)s.%(ext)s'),
        'quiet': False,
        'no_warnings': False,
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Downloading audio from: {instagram_url}")
            info = ydl.extract_info(instagram_url, download=True)
            
            audio_file = os.path.join(output_path, f"{info['id']}.mp3")
            
            return {
                'status': 'success',
                'audio_file': audio_file,
                'video_id': info['id'],
                'title': info.get('title', 'Unknown'),
                'duration': info.get('duration', 0),
                'file_size_mb': os.path.getsize(audio_file) / (1024*1024),
                'downloaded_at': datetime.now().isoformat()
            }
            
    except Exception as e:
        return {
            'status': 'error',
            'error_type': type(e).__name__,
            'message': str(e),
            'url': instagram_url
        }

In [13]:
test_instagram_urls = [
    'https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en',  # Example URL (won't work)
]
 
print("\n" + "=" * 80)
print("TEST 2: Instagram Reel Audio Download (yt-dlp)")
print("=" * 80)

for url in test_instagram_urls:
    print(f"\nAttempting to download: {url}")
    result = download_instagram_reel_audio(url)
    print(json.dumps(result, indent=2))


TEST 2: Instagram Reel Audio Download (yt-dlp)

Attempting to download: https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en
[Instagram] Extracting URL: https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en
[Instagram] DY6pGBLMfbe: Setting up session
[Instagram] DY6pGBLMfbe: Downloading JSON metadata
[info] DY6pGBLMfbe: Downloading 1 format(s): dash-2158729015019780a
[download] Destination: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a
[download] 100% of   82.63KiB in 00:00:00 at 762.65KiB/s 
[FixupM4a] Correcting container of "/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a"
[ExtractAudio] Destination: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3
Deleting original file /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a (pass -k to keep)
{
  "status": "success",
  "audio_file": "/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3

In [7]:
def transcribe_audio_faster_whisper(audio_file: str, model_size: str = 'base') -> Dict:
    """
    Transcribe audio file using faster-whisper.
    
    Args:
        audio_file: Path to audio file
        model_size: Model size ('tiny', 'base', 'small', 'medium', 'large')
        
    Returns:
        Dictionary with transcription and metadata
    """
    
    try:
        from faster_whisper import WhisperModel
        
        print(f"Loading faster-whisper model: {model_size}")
        model = WhisperModel(model_size, device="cpu", compute_type="int8")
        
        print(f"Transcribing: {audio_file}")
        segments, info = model.transcribe(audio_file, language="en")
        
        # Parse segments
        transcript_text = ""
        segments_list = []
        
        for segment in segments:
            segments_list.append({
                'start': segment.start,
                'end': segment.end,
                'text': segment.text,
                'duration': segment.end - segment.start
            })
            transcript_text += segment.text + " "
        
        return {
            'status': 'success',
            'method': 'faster-whisper',
            'audio_file': audio_file,
            'model_size': model_size,
            'transcript': transcript_text.strip(),
            'segments': segments_list,
            'duration_seconds': info.duration,
            'language': info.language,
            'language_probability': info.language_probability,
            'transcribed_at': datetime.now().isoformat()
        }
        
    except ImportError:
        return {
            'status': 'error',
            'error_type': 'DependencyMissing',
            'message': 'faster-whisper not installed. Install with: pip install faster-whisper'
        }
    except Exception as e:
        return {
            'status': 'error',
            'error_type': type(e).__name__,
            'message': str(e),
            'audio_file': audio_file
        }
 
# Demo (without actual file)
print("\n" + "=" * 80)
print("TEST 3: Audio Transcription (faster-whisper)")
print("=" * 80)
print("\nfaster-whisper models available:")
print("  - tiny: Fastest, ~39M parameters")
print("  - base: Fast, ~74M parameters (RECOMMENDED)")
print("  - small: Balanced, ~244M parameters")
print("  - medium: Accurate, ~769M parameters")
print("  - large: Most accurate, ~1.5B parameters")
print("\nNote: Model will be auto-downloaded on first use (~100-1500MB)")


TEST 3: Audio Transcription (faster-whisper)

faster-whisper models available:
  - tiny: Fastest, ~39M parameters
  - base: Fast, ~74M parameters (RECOMMENDED)
  - small: Balanced, ~244M parameters
  - medium: Accurate, ~769M parameters
  - large: Most accurate, ~1.5B parameters

Note: Model will be auto-downloaded on first use (~100-1500MB)


In [14]:
print(transcribe_audio_faster_whisper('/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3'))

Loading faster-whisper model: base
Transcribing: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3
{'status': 'success', 'method': 'faster-whisper', 'audio_file': '/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3', 'model_size': 'base', 'transcript': "Come on, don't leave me  You can't be that easy, baby  If you believe me, I guess I'll", 'segments': [{'start': 0.0, 'end': 2.0, 'text': " Come on, don't leave me", 'duration': 2.0}, {'start': 2.0, 'end': 5.0, 'text': " You can't be that easy, baby", 'duration': 3.0}, {'start': 5.0, 'end': 8.0, 'text': " If you believe me, I guess I'll", 'duration': 3.0}], 'duration_seconds': 8.43025, 'language': 'en', 'language_probability': 1, 'transcribed_at': '2026-05-31T03:00:24.504236'}


In [16]:
def extract_transcript_with_fallback(url: str, prefer_youtube_api: bool = True) -> Dict:
    """
    Smart transcript extraction with fallback strategy.
    
    Flow:
    1. Try youtube-transcript-api (fastest)
    2. Fallback to yt-dlp + faster-whisper (most reliable)
    3. Return best result or error
    """
    
    result = {
        'url': url,
        'timestamp': datetime.now().isoformat(),
        'methods_tried': [],
        'final_result': None
    }
    
    # Determine platform
    is_youtube = 'youtube.com' in url or 'youtu.be' in url
    is_instagram = 'instagram.com' in url
    
    # YouTube extraction
    if is_youtube:
        result['platform'] = 'youtube'
        
        # Method 1: Direct API
        result['methods_tried'].append('youtube-transcript-api')
        transcript_result = extract_youtube_transcript_direct(url)
        
        if transcript_result['status'] == 'success':
            result['final_result'] = transcript_result
            result['final_result']['fallback_used'] = False
            return result
        
        # Method 2: Fallback to yt-dlp + whisper
        result['methods_tried'].append('yt-dlp + faster-whisper')
        print(f"⚠️ YouTube API failed, trying yt-dlp + faster-whisper fallback...")
        
        download_result = download_instagram_reel_audio(url)  # Works with YouTube too
        if download_result['status'] == 'success':
            whisper_result = transcribe_audio_faster_whisper(download_result['audio_file'])
            if whisper_result['status'] == 'success':
                result['final_result'] = whisper_result
                result['final_result']['fallback_used'] = True
                # Clean up audio file
                try:
                    os.remove(download_result['audio_file'])
                except:
                    pass
                return result
    
    # Instagram extraction
    elif is_instagram:
        result['platform'] = 'instagram'
        result['methods_tried'].append('yt-dlp + faster-whisper')
        
        download_result = download_instagram_reel_audio(url)
        if download_result['status'] == 'success':
            whisper_result = transcribe_audio_faster_whisper(download_result['audio_file'])
            if whisper_result['status'] == 'success':
                result['final_result'] = whisper_result
                result['final_result']['fallback_used'] = False
                # Clean up
                try:
                    os.remove(download_result['audio_file'])
                except:
                    pass
                return result
    
    # All methods failed
    result['final_result'] = {
        'status': 'failed',
        'message': 'All extraction methods failed',
        'url': url
    }
    
    return result

In [17]:
extract_transcript_with_fallback('https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en')

[Instagram] Extracting URL: https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en
[Instagram] DY6pGBLMfbe: Setting up session
[Instagram] DY6pGBLMfbe: Downloading JSON metadata
[info] DY6pGBLMfbe: Downloading 1 format(s): dash-2158729015019780a
[download] Destination: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a
[download] 100% of   82.63KiB in 00:00:00 at 771.21KiB/s 
[FixupM4a] Correcting container of "/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a"
[ExtractAudio] Destination: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3
Deleting original file /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.m4a (pass -k to keep)
Loading faster-whisper model: base
Transcribing: /Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3


{'url': 'https://www.instagram.com/reels/DY6pGBLMfbe/?hl=en',
 'timestamp': '2026-05-31T03:03:06.844960',
 'methods_tried': ['yt-dlp + faster-whisper'],
 'final_result': {'status': 'success',
  'method': 'faster-whisper',
  'audio_file': '/Users/gururajgurram/Projects/CODE/Video-Comparision-Chatbot/tmp/DY6pGBLMfbe.mp3',
  'model_size': 'base',
  'transcript': "Come on, don't leave me  You can't be that easy, baby  If you believe me, I guess I'll",
  'segments': [{'start': 0.0,
    'end': 2.0,
    'text': " Come on, don't leave me",
    'duration': 2.0},
   {'start': 2.0,
    'end': 5.0,
    'text': " You can't be that easy, baby",
    'duration': 3.0},
   {'start': 5.0,
    'end': 8.0,
    'text': " If you believe me, I guess I'll",
    'duration': 3.0}],
  'duration_seconds': 8.43025,
  'language': 'en',
  'language_probability': 1,
  'transcribed_at': '2026-05-31T03:03:09.639557',
  'fallback_used': False},
 'platform': 'instagram'}